# Exploration d'espaces latents pour la prédiction d'intervalles de redshift

**Groupe 5 :** Donia ALFONSI, Gabriel Estévez Meye, Anthony Miranda

**Plan :**
1. Présentation et compréhension des données
2. Exploration et analyse descriptive
3. Visualisation de l'espace latent
4. Prétraitement et préparation pour la modélisation
5. Conclusion et limites

VERSION NON FINI !!

In [ ]:
# Bibliothèques utilisées
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor

np.random.seed(42)

---
# 1. Présentation et compréhension des données

## 1.1 Contexte

A compléter :
- définir le redshift
- pourquoi veut-on l'estimer
- pourquoi prédire un **intervalle** et pas une seule valeur
- ...

## 1.2 D'où viennent les données


Images de galaxies simulées  →  CNN entraîné à prédire z  →  espace latent (96 nombres par galaxie)  →  notre travail


A completer avec les autres infos...
comme qui a produit ces données, ou qu'est-ce qui a été fait en amont, ou bien qu'est-ce qui reste à faire de notre côté ???

## 1.3 Chargement des données

In [ ]:
CHEMIN_TRAIN = "../data/epoch_40_train_results.npy"
CHEMIN_EVAL = "../data/epoch_40_val_results.npy"

CLE_X = "final_latent"
CLE_Y = "z_true"

def charger(chemin):
    d = np.load(chemin, allow_pickle=True).item()
    x = np.asarray(d[CLE_X], dtype=np.float64)
    y = np.asarray(d[CLE_Y], dtype=np.float64).ravel()
    return x, y

x_train, y_train = charger(CHEMIN_TRAIN)
x_eval, y_eval = charger(CHEMIN_EVAL)

## 1.4 Structure du jeu de données

In [ ]:
print("Entraînement : x =", x_train.shape, "| y =", y_train.shape)
print("Évaluation   : x =", x_eval.shape, " | y =", y_eval.shape)
print("Type des données :", x_train.dtype)

**Nous allons travailler sur 154 471 galaxies et 1024 variables explicatives (espaces latents).**
Chacune de ces variables est obtenue après un premier traitement réalisé à partir d'un réseau de neurone, c'est pourquoi elles sont toutes de type float.

La variable cible, soit le redshift, est également de type float.

## 1.5 Signification des variables

- *La **variable cible z** : le redshift, qui a un sens physique (c'est une distance)*.
- Les **1024 dimensions latentes** : des nombres produits par le CNN avant notre intervention, sans sens physique propre.
  Ce sont des caractéristiques abstraites que le réseau a jugées utiles pour prédire z.

---
# 2. Exploration et analyse descriptive

## 2.1 Qualité des données
On vérifie les valeurs manquantes, infinies, les doublons et les redshifts impossibles.

In [ ]:
for nom, x, y in [("Entraînement", x_train, y_train), ("Évaluation", x_eval, y_eval)]:
    print(f"--- {nom} ---")
    print("Valeurs manquantes dans x :", np.isnan(x).sum())
    print("Valeurs infinies dans x   :", np.isinf(x).sum())
    print("Valeurs manquantes dans y :", np.isnan(y).sum())
    print("Redshifts négatifs ou nuls:", (y <= 0).sum())
    print("Lignes en double dans x   :", len(x) - len(np.unique(x, axis=0)))

Comme attendu, les espaces latents utilisés sont déjà nettoyés et cohérents.
Cependant, nous pouvons remarquer que dans le jeu d'entrainement il y a 7 doublons.

Regardons de plus près les **7 doublons**.

In [ ]:
groupes = {}
for i in range(len(x_train)):
    cle = tuple(x_train[i])
    if cle not in groupes:
        groupes[cle] = []
    groupes[cle].append(i)

# On garde seulement les vecteurs qui apparaissent plusieurs fois
doublons = []
for cle in groupes:
    if len(groupes[cle]) > 1:
        doublons.append(groupes[cle])

print("Nombre de vecteurs repetes :", len(doublons))

nb_lignes = 0
for d in doublons:
    nb_lignes = nb_lignes + len(d)
print("Nombre de lignes concernees :", nb_lignes)

# Nombre de groupe de n...
tailles = {}
for d in doublons:
    t = len(d)
    if t in tailles:
        tailles[t] = tailles[t] + 1
    else:
        tailles[t] = 1
print("Taille des groupes :", tailles)

meme_z = 0
z_diff = 0
for d in doublons:
    z_groupe = y_train[d]
    if np.max(z_groupe) - np.min(z_groupe) == 0:
        meme_z = meme_z + 1
    else:
        z_diff = z_diff + 1
print("Groupes avec le meme z :", meme_z)
print("Groupes avec des z differents :", z_diff, "\n")

# Affichage
for d in doublons[:]:
    print(f"lignes :{d[:]}   z : {y_train[d]}")

**On remarque que dans chaque groupe de doublons, toutes les lignes ont la même valeur de redshift z**. Deux explications sont possibles :

- Ce sont de vrais doublons : la même galaxie a été enregistrée plusieurs fois, par exemple lors de la simulation ou de la sauvegarde des données
- Ce sont des galaxies différentes : leurs images sont alors identiques, donc le CNN leur donne la même représentation et elles ont le même redshift (très peu probable)

Ainsi, les données sont de bonne qualité : il n'y a pas de valeurs manquantes, ni de valeurs infinies, et aucun redshift impossible (négatif ou nul). Le nettoyage se limite donc à la suppression des doublons.
Ce résultat est cohérent avec l’origine des données qui ont été produites par un CNN, donc déjà préparées en amont.

In [ ]:
a_supprimer = []
for d in doublons:
    a_supprimer = a_supprimer + d[1:]

x_train = np.delete(x_train, a_supprimer, axis=0)
y_train = np.delete(y_train, a_supprimer)

## 2.2 La variable cible : le redshift

In [ ]:
print(pd.concat([pd.Series(y_train, name="train").describe(),
                 pd.Series(y_eval, name="eval").describe()], axis=1))

plt.figure(figsize=(8, 4))
plt.hist(y_train, bins=100, density=True, alpha=0.6, label="Entraînement")
plt.hist(y_eval, bins=100, density=True, alpha=0.6, label="Évaluation")
plt.xlabel("Redshift z")
plt.ylabel("Densité")
plt.title("Distribution du redshift")
plt.legend()
plt.show()

On remarque que les deux jeux de données n'ont **pas la même distribution de redshift z**.

Le jeu d’entraînement a une distribution asymétrique : la majorité des galaxies se situe à bas redshift, avec un maximum vers z ≈ .75, puis une longue traîne vers les grands redshifts, jusqu’à z ≈ 5.34. Les galaxies lointaines sont donc beaucoup moins nombreuses.
75% des galaxies ont un redshift inferieur ou égale à 1.73.

Pour le jeu d’évaluation, c’est différent : les redshifts vont seulement jusqu’à z ≈ 2.52, donc il n’y a pas de galaxies lointaines dedans. On voit aussi beaucoup de pics sur l’histogramme, alors que pour l’entraînement la courbe est assez lisse.

## 2.3 Les espaces latents

In [ ]:
moyennes = x_train.mean(axis=0)
ecarts_types = x_train.std(axis=0)

print("Moyennes des dimensions     : de", moyennes.min().round(3), "à", moyennes.max().round(3))
print("Écarts-types des dimensions : de", ecarts_types.min().round(3), "à", ecarts_types.max().round(3))

print("nombre de dimensions :", x_train.shape[1])

plt.figure(figsize=(14, 4))
plt.boxplot(x_train, showfliers=False)
plt.xlabel("Numéro de la dimension")
plt.ylabel("Valeur")
plt.title("Distribution de chaque dimension latente")
plt.xticks([])
plt.show()

Sur ce graphique, chaque trait correspond au boxplot d'une dimension latente (1024 au total).

On voit que pour presque toutes les dimensions, le boxplot est près de 0, et que la médiane est elle aussi proche de 0. Par contre, certains boxplots montent assez haut. Ça veut dire que pour la plupart des galaxies, les valeurs sont proches de 0, et que seules quelques galaxies ont des valeurs plus grandes. Les distributions sont donc asymétriques.

Les valeurs ne sont jamais négatives et il y a beaucoup de zéros. Cela doit venir de la fonction ReLU utilisée dans le CNN.

## 2.4 Relations avec la cible : quelles dimensions portent l'information ?

In [ ]:
corr_z = np.array([np.corrcoef(x_train[:, j], y_train)[0, 1] for j in range(x_train.shape[1])])

plt.figure(figsize=(14, 4))
plt.bar(range(len(corr_z)), corr_z)
plt.xlabel("Numéro de la dimension")
plt.ylabel("Corrélation avec z")
plt.title("Corrélation de chaque dimension latente avec le redshift")
plt.show()

print("Les 5 dimensions les plus corrélées à z :", np.argsort(-np.abs(corr_z))[:5])

On remarque tout d’abord que les corrélations entre les variables explicatives et le redshift varient entre -0,6 et 0,6. De plus, la grande majorité de ces variables ont une corrélation entre -0,25 et 0,2. Donc, aucune dimension n’est vraiment très liée au redshift à elle seule. L’information sur le redshift serait alors répartie sur plusieurs dimensions comme attendu.

## 2.5 ACP

On va faire une **ACP sur les rangs** car nos données ont quelques valeurs très grandes.

In [ ]:
x_rangs = rankdata(x_train, axis=0) / len(x_train)

acp_rangs = PCA().fit(StandardScaler().fit_transform(x_rangs))
variance_cumulee = np.cumsum(acp_rangs.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(variance_cumulee) + 1), variance_cumulee, "o-", markersize=3)
plt.axhline(0.9, color="red", linestyle="--", label="90 %")
plt.xlabel("Nombre de composantes")
plt.ylabel("Variance expliquée cumulée")
plt.title("ACP sur les rangs : variance expliquée cumulée")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Composantes pour 90 % de la variance :", np.argmax(variance_cumulee >= 0.9) + 1)

La courbe monte rapidement au début, puis elle s’aplatit. Il faut **238 composantes** pour garder 90 % de la variance des données. L’information est donc répartie sur beaucoup de dimensions.

Même si on a divisé le nombre de dimensions par environ 4 (de 1024 à 238), ça reste beaucoup trop pour pouvoir les représenter visuellement.

---
# 3. Visualisation de l'espace latent

In [ ]:
echantillon = np.random.choice(len(y_train), size=min(10000, len(y_train)), replace=False)
X_acp2 = acp_rangs.transform(StandardScaler().fit_transform(x_rangs))[:, :2]

plt.figure(figsize=(7, 6))
nuage = plt.scatter(X_acp2[echantillon, 0], X_acp2[echantillon, 1],
                    c=y_train[echantillon], cmap="viridis", s=3, alpha=0.6)
plt.colorbar(nuage, label="Redshift z")
plt.xlabel("Composante principale 1 (rangs)")
plt.ylabel("Composante principale 2 (rangs)")
plt.title("Les galaxies dans le plan des 2 premières composantes de l'ACP sur les rangs")
plt.show()

Sur ce graphique, on voit que les couleurs sont plutôt mélangées. On remarque tout de même que les galaxies proches sont plutôt d’un côté et les lointaines de l’autre, respéctivement les côtés gauche et droit. Le nuage a une forme ronde et creuse. Donc les 2 premières composantes de l’ACP semblent assez liées au redshift.

Ajouter l'UMAP ici ??

---
# 4. Prétraitement et préparation pour la modélisation

## 4.1 Blanchiment

In [ ]:
moyenne_train = x_train.mean(axis=0)
x_train_c = x_train - moyenne_train
x_eval_c = x_eval - moyenne_train

valeurs, vecteurs = np.linalg.eigh(np.cov(x_train_c, rowvar=False))
valeurs = np.maximum(valeurs, 1e-8 * valeurs.max())
sigma_inv_half = vecteurs @ np.diag(1 / np.sqrt(valeurs)) @ vecteurs.T

z_train = x_train_c @ sigma_inv_half

## 4.2 Création de nouvelles variables : la SIR

La SIR (régression inverse par tranches) cherche les combinaisons des dimensions latentes
les plus liées au redshift :
1. on découpe z en H tranches de même effectif ;
2. on calcule la moyenne des données blanchies dans chaque tranche ;
3. les directions où ces moyennes changent le plus d'une tranche à l'autre sont les plus liées à z.

In [ ]:
# Découpe y en H tranches avec les quantiles
def faire_tranches(y, H):
    tranches = []
    for i in range(H):
        bas = np.quantile(y, i / H)
        haut = np.quantile(y, (i + 1) / H)
        if i == H - 1:
            masque = (y >= bas) & (y <= haut)
        else:
            masque = (y >= bas) & (y < haut)
        tranches.append(np.where(masque)[0])
    return tranches

# Matrice de la SIR 
def calcul_M(z, tranches):
    n = len(z)
    M = np.zeros((z.shape[1], z.shape[1]))
    for t in tranches:
        if len(t) > 0:
            m_h = z[t].mean(axis=0)
            colonne = m_h.reshape(-1, 1)
            M = M + len(t) / n * (colonne @ colonne.T)
    return M

# Métrique
def sigma_mad(z_pred, z_vrai):
    dz = (z_pred - z_vrai) / (1 + z_vrai)
    return 1.4826 * np.median(np.abs(dz - np.median(dz)))

## 4.3 Création des tranches et directions de la SIR

Nous découpons z en H = 40 tranches contenant chacune **le même nombre de galaxies (environ 3 861)**. Comme la distribution du redshift est très asymétrique, avec peu de galaxies lointaines, un petit nombre de tranches regrouperait tous les grands redshifts dans une seule tranche très large.

In [ ]:
H = 40
print(y_train.shape)
tranches = faire_tranches(y_train, H)

seuils = [np.quantile(y_train, i / H) for i in range(1, H)]
print("Seuils :", np.round(seuils, 3), "\n")

for i, t in enumerate(tranches):
    z_min = y_train[t].min()
    z_max = y_train[t].max()
    p_h = len(t) / len(y_train)
    print(f"Tranche {i + 1} : {len(t)} galaxies ({p_h:.1%}), "
          f"z entre {z_min:.3f} et {z_max:.3f}, largeur = {z_max - z_min:.3f}")

Avec **40 tranches**, la dernière tranche couvre une plage plus réduite (de z ≈ 3.474 à 5.343), tout en gardant assez de galaxies par tranche pour calculer des moyennes fiables.

## 4.4 Choix du nombre de variables d

In [ ]:
# Directions de la SIR
vp, eta = np.linalg.eigh(calcul_M(z_train, tranches))
vp, eta = vp[::-1], eta[:, ::-1]

liste_d = [1, 2, 3, 4, 5, 6, 8, 10]
dispersion = []

for d_test in liste_d:
    recette_test = sigma_inv_half @ eta[:, :d_test]
    modele = HistGradientBoostingRegressor(random_state=42).fit(x_train_c @ recette_test, y_train)
    z_pred = modele.predict(x_eval_c @ recette_test)
    dispersion.append(sigma_mad(z_pred, y_eval))
    print("d =", d_test, "| sigma_MAD =", round(dispersion[-1], 4))

# Référence : le même modèle avec toutes les dimensions
modele_ref = HistGradientBoostingRegressor(random_state=42).fit(x_train, y_train)
dispersion_ref = sigma_mad(modele_ref.predict(x_eval), y_eval)
print("Toutes les dimensions | sigma_MAD =", round(dispersion_ref, 4))

plt.figure(figsize=(7, 4))
plt.plot(liste_d, dispersion, "o-", label="SIR")
plt.axhline(dispersion_ref, color="gray", linestyle="--", label="Toutes les dimensions")
plt.yscale("log")
plt.xlabel("Nombre de variables SIR")
plt.ylabel("sigma_MAD (log)")
plt.title("Dispersion selon le nombre de variables")
plt.xticks(liste_d)
plt.grid(alpha=0.3, which="both")
plt.legend()
plt.show()

**ALERT !! Résultats très bizzare... pas normal ? -> problème de traitement des données ? normal ?**

## 4.5 Construction des d nouvelles variables

Les nouvelles variables sont construites à partir du jeu d’entraînement, puis on utilise la même transformation pour le jeu d’évaluation. Chaque nouvelle variable est une somme pondérée des dimensions latentes.

In [ ]:
d = 3
transformation = sigma_inv_half @ eta[:, :d]

X_sir_train = x_train_c @ transformation
X_sir_eval = x_eval_c @ transformation

print("Nouvelles variables : train =", X_sir_train.shape, "| eval =", X_sir_eval.shape)

### Les nouvelles variables et le redshift

Les nouvelles variables n'ont **pas de sens physique** (comme pour l'ACP). On les interprète à travers leur lien avec z.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for k, ax in enumerate(axes):
    hb = ax.hexbin(X_sir_eval[:, k], y_eval, gridsize=60, bins="log", cmap="viridis")
    ax.set_xlabel(f"Variable SIR {k + 1}")
    ax.set_ylabel("Redshift z")
    ax.set_title(f"Variable SIR {k + 1} contre le redshift (évaluation)")
fig.colorbar(hb, ax=axes, label="Nombre de galaxies (log)")
plt.show()

Sur les 3 graphiques, on voit que la plupart des galaxies sont regroupées dans une bande (en jaune). Ça veut dire que **chaque variable SIR est bien liée au redshift**.

On remarque également que ces bandes ne sont pas droites, elles font des virages. Ainsi, pour une même valeur de la variable, il peut y avoir plusieurs redshifts possibles. Avec une seule variable, on ne peut donc pas savoir exactement quel est le redshift. Il faut utiliser les 3 variables ensemble. On voit aussi quelques galaxies isolées, le modèle risque de se tromper sur leur redshift.

Comme les bandes ne sont pas linéaire, cela confirme qu'il faudra utiliser un modèle **non-linéaire** pour les prédictions.

## 4.8 Comparaison SIR / ACP

In [ ]:
# Calculs des rangs
x_eval_rangs = np.zeros(x_eval.shape)
for j in range(x_train.shape[1]):
    colonne_triee = np.sort(x_train[:, j])
    gauche = np.searchsorted(colonne_triee, x_eval[:, j], side="left")
    droite = np.searchsorted(colonne_triee, x_eval[:, j], side="right")
    x_eval_rangs[:, j] = (gauche + droite + 1) / 2 / len(x_train)

# Même ACP sur les rangs que dans l'exploration
normaliseur_rangs = StandardScaler().fit(x_rangs)
X_acp_eval = acp_rangs.transform(normaliseur_rangs.transform(x_eval_rangs))[:, :2]

ech_eval = np.random.choice(len(y_eval), size=min(10000, len(y_eval)), replace=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (nom, Xp) in zip(axes, [("SIR", X_sir_eval), ("ACP sur les rangs", X_acp_eval)]):
    nuage = ax.scatter(Xp[ech_eval, 0], Xp[ech_eval, 1], c=y_eval[ech_eval], cmap="viridis", s=3, alpha=0.6)
    ax.set_title(f"{nom} : deux premières variables")
    ax.set_xlabel("Variable 1")
    ax.set_ylabel("Variable 2")
fig.colorbar(nuage, ax=axes, label="Redshift z")
plt.show()

Les couleurs sont mieux rangées avec la SIR, ça montre qu’une méthode qui se sert du redshift garde mieux ce qui est utile pour le prédire qu’une méthode qui garde seulement ce qui varie le plus, comme l’ACP.

Et puis je ne sais plus quoi dire je suis fatiguée.

## 4.9 La réduction perd-elle de l'information ?

...

---
# 5. Conclusion

...